# Kaze で Qwen3.8-27B を動かす

Triton 製の推論エンジン [Kaze](https://github.com/sahenjp/kaze) で、Qwen3.8-27B（hybrid 注意）を
int4 重みで読み込んで生成します。**L4(24GB) か A100(40GB) 以上**を選んでください。


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q --force-reinstall --no-deps git+https://github.com/sahenjp/kaze
!pip install -q torch triton transformers safetensors huggingface_hub

In [ ]:
from huggingface_hub import snapshot_download

path = snapshot_download('Qwen/Qwen3.8-27B')   # 55GB
path

In [ ]:
import time

import torch
from transformers import AutoTokenizer

from kaze.qwen38 import generate, generate_graph, load_qwen38

# 配布物は VLM 形式（model.language_model.*）なので、ローダ側で text 部分だけ取り出します
model = load_qwen38(path, kind='int4', dtype=torch.float16, verbose=True)
print('GPU', round(torch.cuda.memory_allocated() / 1e9, 2), 'GB')

In [ ]:
tok = AutoTokenizer.from_pretrained(path)

messages = [{'role': 'user', 'content': '深海の調査船について、200字ほどで説明してください。'}]
try:    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                   enable_thinking=False)except Exception:    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
ids = tok(text).input_ids

t0 = time.perf_counter()
out = generate(model, ids, max_tokens=128, temperature=0.0,
               eos_token_ids=(tok.eos_token_id,))
dt = time.perf_counter() - t0
print(tok.decode(out, skip_special_tokens=True))
print(f'--- {len(out)} tokens in {dt:.1f}s ({len(out)/dt:.1f} tok/s)')

## 3. CUDA Graph で速くする（5.7 倍）


In [ ]:
t0 = time.perf_counter()
out2 = generate_graph(model, ids, max_tokens=64, max_len=len(ids)+128,
                      eos_token_ids=(tok.eos_token_id,))
dt2 = time.perf_counter() - t0
print(tok.decode(out2, skip_special_tokens=True))
print(f'--- graph: {len(out2)} tokens in {dt2:.1f}s ({len(out2)/dt2:.1f} tok/s)')

In [ ]:
# OpenAI 互換のサーバとしても出せます（別セルで）
# from kaze.qwen38_server import create_app
# create_app(model, tok, graph=True) を uvicorn で起動

## メモ

- decode は 1 トークンずつ Python から回しているため遅いです（CUDA Graph 未対応）。
  A100 80GB で 4 tok/s 前後を実測しています。
- 量子化は読み込み時にその場で行います（変換済みファイルは不要）。
- 視覚側の重み（`model.visual.*`）と MTP は読み飛ばします。
- メモリが足りないときは `kind='int4'` のまま `max_tokens` を小さくしてください。
